In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from config import SimConfig
warnings.filterwarnings('ignore')


In [2]:
# --- 1. PHYSICAL ANCHORS (LOCKED CONSTANTS) ---
cfg = SimConfig()

A0 = cfg.a0               # g/s (Replace with cfg.a0)
A1 = cfg.a1               # g/(s kW) (Replace with cfg.a1)
A2 = cfg.a2               # g/(s kW^2) (Replace with cfg.a2)
P_MAX = cfg.p_max         # kW (Using 200kW per module for Mariner)
N_MAX = len(cfg.n_vals)   # Max modules

K_FC = 750 * P_MAX        # Euro (Replace with cfg.k_fc)
LHV = cfg.LHV             # kWh/kg

EPS = 1e-9
K_E = (3600 * LHV) / 1000


In [3]:
# ==========================================
# ERROR HANDLER
# ==========================================
def show_error_screen(message):
    fig = go.Figure()
    fig.add_annotation(text=f"<b>PHYSICS VIOLATION</b><br><br>{message}", x=0.5, y=0.5, showarrow=False, font=dict(color="red", size=24))
    fig.update_layout(xaxis=dict(visible=False), yaxis=dict(visible=False), plot_bgcolor='rgba(255, 230, 230, 1)', height=750, width=1350)
    fig.show()


In [4]:
# ==========================================
# LEVEL 1: THERMODYNAMIC FEASIBILITY
# ==========================================
def plot_level_1(p_nom):
    p_opt_floor = np.sqrt(A0 / A2)
    p_opt_range = np.linspace(p_opt_floor + 0.1, p_nom - 0.1, 250)
    
    d1 = K_E * (A1 + 2 * A2 * p_opt_range)
    eta_floor_curve = 1.0 / d1
    d2 = K_E * ((A1 + 2 * A2 * p_nom) * (p_nom + p_opt_range) - 2 * (A2 * p_nom**2 - A0))
    eta_ceil_curve = (p_nom + p_opt_range) / d2
    
    eta_min, eta_max = np.nanmin(eta_floor_curve), np.nanmax(eta_ceil_curve)
    eta_opt_range = np.linspace(eta_min, eta_max, 250)
    P_OPT, ETA_OPT = np.meshgrid(p_opt_range, eta_opt_range)
    
    K_eff = 3600 * LHV * ETA_OPT
    D = A1 + 2 * p_nom * A2
    denom = (K_eff * D) - 1000
    
    M = np.where(np.abs(denom) > 1e-9, (2000 * (p_nom - P_OPT)) / denom, np.nan)
    X_const = (M * K_eff) / 1000
    
    num = (1 - A2 * X_const) * (p_nom**2)
    den = (P_OPT**2) - (p_nom**2) - X_const * (A0 - A2 * p_nom**2)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        ALPHA = num / den
        
    valid_mask = (den > 0) & (num > 0) & (ALPHA > 0)
    ALPHA = np.where(valid_mask, ALPHA, np.nan)
    
    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=ALPHA, x=p_opt_range, y=eta_opt_range, colorscale='Viridis', zmin=0, zmax=3.0,
        colorbar=dict(title=r'Degradation (α)', xpad=10),
        hovertemplate='p_opt: %{x:.1f} kW<br>eta_opt: %{y:.3f}<br>Alpha: %{z:.2f}<extra></extra>'
    ))
    fig.update_layout(
        title=dict(text=f"<b>Level 1: Thermodynamic Feasibility Envelope</b><br><span style='font-size:14px'>Nominal Cruising Baseline: p_nom = {p_nom} kW</span>", x=0.5, xanchor='center'),
        xaxis_title="Optimal Power (p_opt) [kW]", yaxis_title="Target Peak Efficiency (η_opt)",
        height=750, width=1350, plot_bgcolor='rgba(230, 230, 230, 1)'
    )

    display(fig)


In [5]:
# ==========================================
# LEVEL 2: HARDWARE TRANSLATION
# ==========================================
def plot_level_2(p_nom, p_opt, eta_opt, target_kh2, target_tau, target_smax):
    K_eff = 3600 * LHV * eta_opt
    D = A1 + 2 * p_nom * A2
    denom = K_eff * D - 1000
    
    if abs(denom) < 1e-9:
        return show_error_screen("Mathematical Singularity Reached.")
    M = (2000 * (p_nom - p_opt)) / denom
    X_const = (M * K_eff) / 1000
    Y_const = (p_opt**2) - (p_nom**2) - X_const * (A0 - A2 * p_nom**2)
    
    num = (1 - A2 * X_const) * p_nom**2
    if Y_const <= 0 or num <= 0 or M <= 0:
        return show_error_screen("Configuration results in negative degradation or infinite lifespan.")
    alpha = num / Y_const

    # Target Lambda 1 Calculations
    lambda1_k = target_kh2 / (M * K_eff)
    lambda1_tau = K_FC / (3600 * Y_const * target_tau)
    lambda2_k = K_FC / (lambda1_k * target_smax)
    lambda2_tau = K_FC / (lambda1_tau * target_smax)

    l1_min, l1_max = min(lambda1_k, lambda1_tau), max(lambda1_k, lambda1_tau)
    lambda1_range = np.logspace(np.log10(l1_min) - 1, np.log10(l1_max) + 1, 200)
    l1_center = 10**((np.log10(l1_min) + np.log10(l1_max)) / 2)
    l2_center = K_FC / (l1_center * target_smax)
    lambda2_range = np.logspace(np.log10(l2_center) - 1.5, np.log10(l2_center) + 1.5, 200)

    k_h2_line = (M * K_eff) * lambda1_range
    tau_fc_line = K_FC / (3600 * Y_const * lambda1_range)
    L1, L2 = np.meshgrid(lambda1_range, lambda2_range)
    S_MAX = K_FC / (L1 * L2)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("1. Hydrogen Price Convergence", "2. Stack Lifetime Convergence", "3. Mechanical Limit (S_max)", "4. Heuristic Stubbornness (k_s)"),
        vertical_spacing=0.15,
        horizontal_spacing=0.18
    )

    fig.add_trace(go.Scatter(x=lambda1_range, y=k_h2_line, line=dict(color='black', width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=lambda1_range, y=tau_fc_line, line=dict(color='black', width=3)), row=1, col=2)
    fig.add_hline(y=target_kh2, line_dash="dash", line_color="black", row=1, col=1)
    fig.add_hline(y=target_tau, line_dash="dash", line_color="black", row=1, col=2)

    fig.add_trace(go.Heatmap(z=S_MAX, x=lambda1_range, y=lambda2_range, colorscale='Viridis', zmin=1000, zmax=100000, colorbar=dict(x=0.43, y=0.22, len=0.4)), row=2, col=1)
    fig.add_trace(go.Heatmap(z=L1*L2, x=lambda1_range, y=lambda2_range, colorscale='Plasma', zmax=500, colorbar=dict(x=1.0, y=0.22, len=0.4)), row=2, col=2)

    diagonal_smax = K_FC / (lambda1_range * target_smax)
    for col in [1, 2]:
        fig.add_trace(go.Scatter(x=lambda1_range, y=diagonal_smax, mode='lines', line=dict(color='white', width=3, dash='dash')), row=2, col=col)
        fig.add_vline(x=lambda1_k, line_color="#00CC96", line_width=3, line_dash="solid", row=1, col=col)
        fig.add_vline(x=lambda1_k, line_color="#00CC96", line_width=3, line_dash="solid", row=2, col=col)
        fig.add_vline(x=lambda1_tau, line_color="#EF553B", line_width=3, line_dash="solid", row=1, col=col)
        fig.add_vline(x=lambda1_tau, line_color="#EF553B", line_width=3, line_dash="solid", row=2, col=col)

    status_color = "#00CC96" if abs(lambda1_k - lambda1_tau) / lambda1_k < 0.05 else "black"
    msg = "🎯 TARGETS ALIGNED!" if status_color == "#00CC96" else "Adjust p_opt / η_opt to merge lines"
    
    title_html = (
        f"<span style='font-size:16px'>Hardware Anchors: <b>p_nom = {p_nom} kW</b> | Calculated Degradation: <b>α = {alpha:.3f}</b></span><br>"
        f"<b style='font-size:20px; color:{status_color}'>{msg}</b><br>"
        f"<span style='font-size:13px; color:#00CC96'>● Fuel Target (Green): λ₁ = <b>{lambda1_k:.2e}</b> ➔ Requires λ₂ = <b>{lambda2_k:.1e}</b></span> | "
        f"<span style='font-size:13px; color:#EF553B'>● Life Target (Red): λ₁ = <b>{lambda1_tau:.2e}</b> ➔ Requires λ₂ = <b>{lambda2_tau:.1e}</b></span>"
    )
    
    fig.update_layout(title=dict(text=title_html, x=0.5, xanchor='center', y=0.96),
                         height=850, width=1350,
                         showlegend=False,
                         plot_bgcolor='rgba(245, 245, 245, 1)',
                         margin=dict(t=120)
                        )
    for r in [1, 2]:
        for c in [1, 2]:
            fig.update_xaxes(type="log", title_text="λ₁ (OPEX Multiplier)", row=r, col=c)
    fig.update_yaxes(title_text="H2 Price [€/kg]", range=[0, target_kh2 * 2], row=1, col=1)
    fig.update_yaxes(title_text="Lifetime [Hrs]", range=[0, target_tau * 2], row=1, col=2)
    fig.update_yaxes(type="log", title_text="λ₂ (Stubbornness)", row=2, col=1)
    fig.update_yaxes(type="log", title_text="λ₂ (Stubbornness)", row=2, col=2)

    display(fig)


In [6]:
# ==========================================
# LEVEL 3: CONTROLLER BEHAVIOR & HEURISTICS
# ==========================================
def plot_level_3(p_nom, p_opt, eta_opt, lambda_1, lambda_2):
    # Hardware Math
    K_eff = 3600 * LHV * eta_opt
    D = A1 + 2 * p_nom * A2
    denom = K_eff * D - 1000
    
    if abs(denom) < 1e-9:
        return show_error_screen("Mathematical Singularity Reached.")
    M = (2000 * (p_nom - p_opt)) / denom
    X_const = (M * K_eff) / 1000
    Y_const = (p_opt**2) - (p_nom**2) - X_const * (A0 - A2 * p_nom**2)
    num = (1 - A2 * X_const) * p_nom**2
    if Y_const <= 0 or num <= 0 or M <= 0:
        return show_error_screen("Configuration results in negative degradation or infinite lifespan.")
    
    # Calculate specific hardware outputs based on the chosen lambdas
    k_h2_val = (M * K_eff) * lambda_1
    tau_fc_val = K_FC / (3600 * Y_const * lambda_1)
    s_max_val = K_FC / (lambda_1 * lambda_2)
    k_s_val = lambda_1 * lambda_2
    
    # Macroscopic Controller Equations (SDP Inputs)
    A = lambda_1
    C = (p_opt**2) * lambda_1
    B = (M - 2 * p_opt) * lambda_1
    
    # Ship Power Demand Sweep (Up to max total power)
    P_d = np.linspace(10, P_MAX * N_MAX, 500) 
    n_modules = np.arange(1, N_MAX + 1)
    
    # Evaluate Cost Curve C_o(n, P_d)
    costs = np.zeros((len(n_modules), len(P_d)))
    for i, n in enumerate(n_modules):
        costs[i, :] = A * (P_d**2 / n) + B * P_d + n * C
        
    n_opt_idx = np.argmin(costs, axis=0)
    n_opt = n_modules[n_opt_idx]
    c_opt = np.min(costs, axis=0)
    
    # Prepare Subplots using Plotly
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("1. Ideal Static Policy", "2. Financial Forgiveness (ΔCo %)", "3. Break-Even Time (T_be)"),
        horizontal_spacing=0.08
    )
    
    # Plot 1: Optimal Active Modules
    fig.add_trace(go.Scatter(x=P_d, y=n_opt, mode='lines', line=dict(color='purple', width=2, shape='hv'), name='n_opt'), row=1, col=1)
    
    # Plot 2: Normalized Cost Penalty (Showing just 4, 8, 12, 16 for cleanliness)
    display_n = [4, 8, 12, 16]
    for n in display_n:
        penalty = 100 * (costs[n-1, :] - c_opt) / c_opt
        fig.add_trace(go.Scatter(x=P_d, y=penalty, mode='lines', name=f'n={n}'), row=1, col=2)
        
    # Plot 3: Universal Break-Even Holding Time (Comparing n to n+1 for a few states)
    display_switches = [4, 8, 12]
    for n in display_switches:
        n_new = n + 1
        denom_t = np.abs(P_d**2 * (1/n - 1/n_new) + p_opt**2 * (n - n_new))
        with np.errstate(divide='ignore', invalid='ignore'):
            t_break = np.where(denom_t != 0, lambda_2 / denom_t, np.inf)
        fig.add_trace(go.Scatter(x=P_d, y=t_break, mode='lines', name=f'{n} → {n_new}'), row=1, col=3)
        
    # Dynamic Title with Hardware Values
    title_html = (
        f"<b>Level 3: Controller Behavior & Heuristics</b><br>"
        f"<span style='font-size:14px; color:gray'>Resulting Hardware: H2 Price = <b>{k_h2_val:.2f} €/kg</b> | Life = <b>{tau_fc_val:,.0f} hrs</b> | S_max = <b>{s_max_val:,.0f}</b> | k_s = <b>{k_s_val:.2f} €</b></span>"
    )
    
    fig.update_layout(title=dict(text=title_html, x=0.5, xanchor='center'), height=650, width=1350, plot_bgcolor='rgba(245, 245, 245, 1)', margin=dict(t=100))
    
    # Formatting axes
    fig.update_xaxes(title_text="Ship Power Demand (P_d) [kW]")
    fig.update_yaxes(title_text="Optimal Modules (n)", row=1, col=1)
    fig.update_yaxes(title_text="Penalty (%)", range=[-2, 50], row=1, col=2)
    fig.update_yaxes(title_text="Time (s)", range=[0, 3600], row=1, col=3) # Clipped to 1 hour

    display(fig)


In [7]:
# ==========================================
# WIDGET UI CONSTRUCTION
# ==========================================
style = {'description_width': 'initial'}

# Define All Sliders
w1_p_nom = widgets.FloatSlider(value=80.0, min=60.0, max=100.0, step=1.0, description='p_nom (kW):', style=style)
w2_p_opt = widgets.FloatSlider(value=70.0, min=30.0, max=100.0, step=0.1, description='p_opt (kW):', style=style)
w2_eta_opt = widgets.FloatSlider(value=0.55, min=0.45, max=0.60, step=0.005, description='\u03B7_opt:', style=style, readout_format='.3f')
w2_tgt_kh2 = widgets.FloatSlider(value=4.0, min=1.6, max=16.0, step=0.1, description='Target H2 (€/kg):', style=style)
w2_tgt_tau = widgets.IntSlider(value=50000, min=10000, max=100000, step=5000, description='Target Life (h):', style=style)
w2_tgt_smax = widgets.IntSlider(value=50000, min=5000, max=100000, step=5000, description='Target S_max:', style=style)

# Level 3 Sliders (Logarithmic)
w3_lambda1 = widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=-2, step=0.1, description='λ₁ (OPEX):', style=style)
w3_lambda2 = widgets.FloatLogSlider(value=1e4, base=10, min=2, max=6, step=0.1, description='λ₂ (Stubborn):', style=style)

def update_dynamic_bounds(*args):
    """Calculates and strictly enforces the physical boundaries on the sliders."""
    p_nom = w1_p_nom.value
    
    # Clean Decimal Snap for p_opt (Fixes the .x9 issue)
    raw_floor = (A0 / A2)**0.5
    p_opt_floor = np.ceil(raw_floor * 10) / 10 
    p_opt_ceil = np.floor((p_nom - 0.1) * 10) / 10
    
    if p_opt_ceil > w2_p_opt.max:
         w2_p_opt.max = p_opt_ceil
    w2_p_opt.min = p_opt_floor
    w2_p_opt.max = p_opt_ceil
    
    p_opt = w2_p_opt.value
    
    # ETA Bounds
    d1 = K_E * (A1 + 2 * A2 * p_opt)
    eta_floor = 1.0 / d1 if d1 != 0 else 0.4
    d2 = K_E * ((A1 + 2 * A2 * p_nom) * (p_nom + p_opt) - 2 * (A2 * p_nom**2 - A0))
    eta_ceil = (p_nom + p_opt) / d2 if d2 > 0 else 0.8
    
    eta_floor = max(0.20, min(eta_floor, 0.90))
    eta_ceil = max(eta_floor + 0.01, min(eta_ceil, 0.95))
    
    if eta_ceil > w2_eta_opt.max:
         w2_eta_opt.max = eta_ceil
    w2_eta_opt.min = np.ceil(eta_floor*100)/100
    w2_eta_opt.max = np.floor(eta_ceil*100)/100

    # Update Lambda Bounds based on Tab 2 Targets
    K_eff = 3600 * LHV * w2_eta_opt.value
    D = A1 + 2 * p_nom * A2
    denom = K_eff * D - 1000
    if abs(denom) > 1e-9:
        M = (2000 * (p_nom - p_opt)) / denom
        X_const = (M * K_eff) / 1000
        Y_const = (p_opt**2) - (p_nom**2) - X_const * (A0 - A2 * p_nom**2)
        if Y_const > 0 and M > 0:
            lambda1_k = w2_tgt_kh2.value / (M * K_eff)
            lambda1_tau = K_FC / (3600 * Y_const * w2_tgt_tau.value)
            l1_min, l1_max = min(lambda1_k, lambda1_tau), max(lambda1_k, lambda1_tau)
            
            # Pad lambda bounds
            w3_lambda1.min = np.floor(np.log10(l1_min) - 1)
            w3_lambda1.max = np.ceil(np.log10(l1_max) + 1)
            
            l1_center = 10**((np.log10(l1_min) + np.log10(l1_max)) / 2)
            l2_center = K_FC / (l1_center * w2_tgt_smax.value)
            w3_lambda2.min = np.floor(np.log10(l2_center) - 1.5)
            w3_lambda2.max = np.ceil(np.log10(l2_center) + 1.5)

# Bind observers
w1_p_nom.observe(update_dynamic_bounds, 'value')
w2_p_opt.observe(update_dynamic_bounds, 'value')
w2_eta_opt.observe(update_dynamic_bounds, 'value')
w2_tgt_kh2.observe(update_dynamic_bounds, 'value')
w2_tgt_tau.observe(update_dynamic_bounds, 'value')
w2_tgt_smax.observe(update_dynamic_bounds, 'value')
update_dynamic_bounds() # Initialize safe bounds

# Bind Outputs
out1 = widgets.interactive_output(plot_level_1, {'p_nom': w1_p_nom})
out2 = widgets.interactive_output(plot_level_2, {
    'p_nom': w1_p_nom, 'p_opt': w2_p_opt, 'eta_opt': w2_eta_opt,
    'target_kh2': w2_tgt_kh2, 'target_tau': w2_tgt_tau, 'target_smax': w2_tgt_smax
})
out3 = widgets.interactive_output(plot_level_3, {
    'p_nom': w1_p_nom, 'p_opt': w2_p_opt, 'eta_opt': w2_eta_opt,
    'lambda_1': w3_lambda1, 'lambda_2': w3_lambda2
})

# ==========================================
# 3. CONSTRUCT LAYOUT (Graphs Top, Sliders Bottom)
# ==========================================
center_layout = widgets.Layout(justify_content='center')

# Level 1 Layout
controls_l1 = widgets.HBox([w1_p_nom], layout=center_layout)
out1_centered = widgets.HBox([out1], layout=center_layout)
tab1 = widgets.VBox([out1_centered, controls_l1])

# Level 2 Layout 
target_row = widgets.HBox([w2_tgt_kh2, w2_tgt_tau, w2_tgt_smax], layout=center_layout)
thermo_row = widgets.HBox([w2_p_opt, w2_eta_opt], layout=center_layout)

controls_l2 = widgets.VBox([
    widgets.HTML("<div style='text-align:center'><b>1. Define Hardware Targets (OPEX & CAPEX)</b></div>"), target_row,
    widgets.HTML("<div style='text-align:center'><b>2. Steer Thermodynamics to Align Targets</b></div>"), thermo_row
])
out2_centered = widgets.HBox([out2], layout=center_layout)
tab2 = widgets.VBox([out2_centered, controls_l2])

# Level 3 Layout
lambda_row = widgets.HBox([w3_lambda1, w3_lambda2], layout=center_layout)
controls_l3 = widgets.VBox([
    widgets.HTML("<div style='text-align:center'><b>Test Controller Configurations</b></div>"), lambda_row
])
out3_centered = widgets.HBox([out3], layout=center_layout)
tab3 = widgets.VBox([out3_centered, controls_l3])

# Compile Dashboard
dashboard = widgets.Tab(children=[tab1, tab2, tab3])
dashboard.set_title(0, 'Level 1: Thermodynamics')
dashboard.set_title(1, 'Level 2: Target Solver')
dashboard.set_title(2, 'Level 3: Heuristics')

display(dashboard)
